In [9]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

#==========================================
#LOAD DATA
#==========================================


df = pd.read_excel("dynamic_supply_chain_logistics_dataset.xlsx")
#==========================================
#DROP TARGETS NOT USED
#==========================================

df = df.drop(columns=[
'delay_probability',
'risk_classification'
])

#==========================================
#TARGET VARIABLE
#==========================================

target = 'delivery_time_deviation'

X = df.drop(columns=[target])
y = df[target]

#==========================================
#HANDLE TIMESTAMP
#==========================================

X['timestamp'] = pd.to_datetime(X['timestamp'])

X['year'] = X['timestamp'].dt.year
X['month'] = X['timestamp'].dt.month
X['day'] = X['timestamp'].dt.day
X['hour'] = X['timestamp'].dt.hour

X = X.drop(columns=['timestamp'])

#==========================================
#IDENTIFY COLUMN TYPES
#==========================================

categorical_features = X.select_dtypes(include=['object']).columns
numerical_features = X.select_dtypes(exclude=['object']).columns

#==========================================
#PREPROCESSING
#==========================================

preprocessor = ColumnTransformer(
transformers=[
('cat', OneHotEncoder(handle_unknown='ignore'),
categorical_features),
('num', 'passthrough',
numerical_features)
]
)

#==========================================
#MODEL
#==========================================

model = RandomForestRegressor(
n_estimators=200,
max_depth=15,
random_state=42
)

pipeline = Pipeline([
('preprocessor', preprocessor),
('model', model)
])

#==========================================
#SPLIT DATA
#==========================================

X_train, X_test, y_train, y_test = train_test_split(
X,
y,
test_size=0.2,
random_state=42
)

#==========================================
#TRAIN
#==========================================

pipeline.fit(X_train, y_train)

#==========================================
#PREDICT
#==========================================

predictions = pipeline.predict(X_test)

#==========================================
#EVALUATION
#==========================================

mae = mean_absolute_error(y_test, predictions)

rmse = np.sqrt(mean_squared_error(
y_test,
predictions
))

r2 = r2_score(y_test, predictions)

print("Delivery Delay Prediction Results")
print("MAE :", mae)
print("RMSE:", rmse)
print("R2 :", r2)

#==========================================
#FEATURE IMPORTANCE
#==========================================

feature_names = (
pipeline.named_steps['preprocessor']
.get_feature_names_out()
)

importance = (
pipeline.named_steps['model']
.feature_importances_
)

feature_importance = pd.DataFrame({
'Feature': feature_names,
'Importance': importance
})

feature_importance = feature_importance.sort_values(
by='Importance',
ascending=False
)

print(feature_importance.head(20))

Delivery Delay Prediction Results
MAE : 3.7097029456537807
RMSE: 4.160205033630907
R2 : -0.0036502207034798673
                                 Feature  Importance
9        num__weather_condition_severity    0.044496
6            num__loading_unloading_time    0.044249
20         num__fatigue_monitoring_score    0.044229
5         num__warehouse_inventory_level    0.043191
18           num__customs_clearance_time    0.042991
2             num__fuel_consumption_rate    0.042906
12       num__supplier_reliability_score    0.042903
7   num__handling_equipment_availability    0.042832
3               num__eta_variation_hours    0.041974
0              num__vehicle_gps_latitude    0.041833
17                 num__route_risk_level    0.041729
10            num__port_congestion_level    0.041622
8          num__order_fulfillment_status    0.041510
13                   num__lead_time_days    0.041318
4          num__traffic_congestion_level    0.041196
21      num__disruption_likelihood_score 

In [10]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor

from sklearn.metrics import (
mean_absolute_error,
mean_squared_error,
r2_score
)

#=====================================================
#LOAD DATA
#=====================================================

df = pd.read_excel("dynamic_supply_chain_logistics_dataset.xlsx")

#=====================================================
#TARGET
#=====================================================

target = "delivery_time_deviation"



df = df.drop(columns=[
"delay_probability",
"risk_classification"
])

#=====================================================
#FEATURE ENGINEERING
#=====================================================

df["timestamp"] = pd.to_datetime(df["timestamp"])

df["year"] = df["timestamp"].dt.year
df["month"] = df["timestamp"].dt.month
df["day"] = df["timestamp"].dt.day
df["hour"] = df["timestamp"].dt.hour
df["dayofweek"] = df["timestamp"].dt.dayofweek

df = df.drop(columns=["timestamp"])

#=====================================================
#SPLIT FEATURES / TARGET
#=====================================================

X = df.drop(columns=[target])
y = df[target]

#=====================================================
#CATEGORICAL / NUMERICAL
#=====================================================

categorical_features = X.select_dtypes(
include=["object"]
).columns

numerical_features = X.select_dtypes(
exclude=["object"]
).columns

#=====================================================
#PREPROCESSING
#=====================================================

numeric_transformer = Pipeline([
("imputer", SimpleImputer(strategy="median")),
("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
("imputer", SimpleImputer(strategy="most_frequent")),
("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
transformers=[
("num", numeric_transformer, numerical_features),
("cat", categorical_transformer, categorical_features)
]
)

#=====================================================
#TRAIN TEST SPLIT
#=====================================================

X_train, X_test, y_train, y_test = train_test_split(
X,
y,
test_size=0.20,
random_state=42
)

#=====================================================
#MODELS
#=====================================================

models = {

"Linear Regression":
    LinearRegression(),

"Random Forest":
    RandomForestRegressor(
        n_estimators=300,
        max_depth=20,
        random_state=42,
        n_jobs=-1
    ),

"Neural Network":
    MLPRegressor(
        hidden_layer_sizes=(128,64,32),
        activation="relu",
        solver="adam",
        learning_rate_init=0.001,
        max_iter=200,
        random_state=42
    )

}

#=====================================================
#TRAIN AND EVALUATE
#=====================================================

results = []

for model_name, model in models.items():

  pipeline = Pipeline([ ("preprocessor", preprocessor), ("model", model) ])

  pipeline.fit(X_train, y_train)

  predictions = pipeline.predict(X_test)

  mae = mean_absolute_error(
    y_test,
    predictions
    )

  rmse = np.sqrt(
    mean_squared_error(
        y_test,
        predictions
    )
    )

  r2 = r2_score(
    y_test,
    predictions
    )

  results.append([ model_name, mae, rmse, r2 ])

  print("="*60)
  print(model_name)
  print("MAE :", mae)
  print("RMSE:", rmse)
  print("R2  :", r2)
#=====================================================
#COMPARISON TABLE
#=====================================================

results_df = pd.DataFrame(
results,
columns=[
"Model",
"MAE",
"RMSE",
"R2"
]
)

print("\nFINAL COMPARISON")
print(results_df.sort_values(
by="R2",
ascending=False
))

Linear Regression
MAE : 3.7040964393415177
RMSE: 4.15308558368806
R2  : -0.0002180228301051912
Random Forest
MAE : 3.7143247839695026
RMSE: 4.164319706303778
R2  : -0.005636533565328783
Neural Network
MAE : 4.600579876650153
RMSE: 5.671372428487115
R2  : -0.8652165025132312

FINAL COMPARISON
               Model       MAE      RMSE        R2
0  Linear Regression  3.704096  4.153086 -0.000218
1      Random Forest  3.714325  4.164320 -0.005637
2     Neural Network  4.600580  5.671372 -0.865217


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor

from sklearn.metrics import (
mean_absolute_error,
mean_squared_error,
r2_score
)

#=====================================================
#LOAD DATA
#=====================================================

df = pd.read_excel("dynamic_supply_chain_logistics_dataset.xlsx")

#=====================================================
#TARGET
#=====================================================

target = "delivery_time_deviation"



df = df.drop(columns=[
"delay_probability",
"risk_classification"
])

#=====================================================
#FEATURE ENGINEERING
#=====================================================

df["timestamp"] = pd.to_datetime(df["timestamp"])

df["year"] = df["timestamp"].dt.year
df["month"] = df["timestamp"].dt.month
df["day"] = df["timestamp"].dt.day
df["hour"] = df["timestamp"].dt.hour
df["dayofweek"] = df["timestamp"].dt.dayofweek

df = df.drop(columns=["timestamp"])

#=====================================================
#SPLIT FEATURES / TARGET
#=====================================================

X = df.drop(columns=[target])
y = df[target]

#=====================================================
#CATEGORICAL / NUMERICAL
#=====================================================

categorical_features = X.select_dtypes(
include=["object"]
).columns

numerical_features = X.select_dtypes(
exclude=["object"]
).columns

#=====================================================
#PREPROCESSING
#=====================================================

numeric_transformer = Pipeline([
("imputer", SimpleImputer(strategy="median")),
("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
("imputer", SimpleImputer(strategy="most_frequent")),
("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
transformers=[
("num", numeric_transformer, numerical_features),
("cat", categorical_transformer, categorical_features)
]
)

#=====================================================
#TRAIN TEST SPLIT
#=====================================================

X_train, X_test, y_train, y_test = train_test_split(
X,
y,
test_size=0.20,
random_state=42
)

model = LinearRegression()
pipeline = Pipeline([ ("preprocessor", preprocessor), ("model", model) ])

  pipeline.fit(X_train, y_train)

  predictions = pipeline.predict(X_test)

  mae = mean_absolute_error(
    y_test,
    predictions
    )

  rmse = np.sqrt(
    mean_squared_error(
        y_test,
        predictions
    )
    )

In [9]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =====================================================
# LOAD DATA
# =====================================================

df = pd.read_excel("dynamic_supply_chain_logistics_dataset.xlsx")

# =====================================================
# TARGET
# =====================================================

target = "delivery_time_deviation"

# Drop leaky / pre-derived columns
df = df.drop(columns=["delay_probability", "risk_classification"])

# =====================================================
# FEATURE ENGINEERING
# =====================================================

df["timestamp"] = pd.to_datetime(df["timestamp"])
df["month"]     = df["timestamp"].dt.month
df["day"]       = df["timestamp"].dt.day
df["hour"]      = df["timestamp"].dt.hour
df["dayofweek"] = df["timestamp"].dt.dayofweek
df = df.drop(columns=["timestamp"])

# =====================================================
# SPLIT FEATURES / TARGET
# =====================================================

X = df.drop(columns=[target])
y = df[target]

# =====================================================
# PREPROCESSING
# =====================================================

numerical_features = X.select_dtypes(exclude=["object"]).columns

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numerical_features)
])

# =====================================================
# TRAIN / TEST SPLIT
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# =====================================================
# DIAGNOSTIC: BASELINE (predict mean)
# =====================================================

baseline_pred = np.full(len(y_test), y_train.mean())
print("=" * 60)
print("BASELINE — predict training mean for every sample")
print("MAE :", mean_absolute_error(y_test, baseline_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, baseline_pred)))
print("R2  :", r2_score(y_test, baseline_pred))
print()
print("NOTE: R2 ≈ 0 for the baseline confirms the target is")
print("essentially unpredictable from the available features.")
print("All models score near or below 0 — not a code bug.")
print("=" * 60)

# =====================================================
# MODELS
# =====================================================
# FIX 1: MLPRegressor — added early_stopping and raised
#         max_iter to 1 000 to eliminate ConvergenceWarning.
# FIX 2: Removed 'year' feature (constant in this dataset).
# =====================================================

models = {
    "Linear Regression": LinearRegression(),

    "Ridge Regression": Ridge(alpha=1.0),

    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        max_depth=20,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        random_state=42
    ),

    "Neural Network": MLPRegressor(
        hidden_layer_sizes=(128, 64, 32),
        activation="relu",
        solver="adam",
        learning_rate_init=0.001,
        max_iter=1000,          # was 200 — caused ConvergenceWarning
        early_stopping=True,    # stops when val loss stops improving
        validation_fraction=0.1,
        n_iter_no_change=20,
        random_state=42
    ),
}

# =====================================================
# TRAIN AND EVALUATE
# =====================================================

results = []

for model_name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model",        model)
    ])

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    mae  = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2   = r2_score(y_test, predictions)

    results.append([model_name, mae, rmse, r2])

    print("=" * 60)
    print(model_name)
    print("MAE :", mae)
    print("RMSE:", rmse)
    print("R2  :", r2)

# =====================================================
# COMPARISON TABLE
# =====================================================

results_df = pd.DataFrame(
    results,
    columns=["Model", "MAE", "RMSE", "R2"]
)

print("\nFINAL COMPARISON")
print(results_df.sort_values(by="R2", ascending=False))

BASELINE — predict training mean for every sample
MAE : 3.7047929423130506
RMSE: 4.152659051420654
R2  : -1.2583599518167787e-05

NOTE: R2 ≈ 0 for the baseline confirms the target is
essentially unpredictable from the available features.
All models score near or below 0 — not a code bug.
Linear Regression
MAE : 3.704174504056116
RMSE: 4.153122754274584
R2  : -0.0002359270394880486
Ridge Regression
MAE : 3.704174497841591
RMSE: 4.153122659814626
R2  : -0.00023588154011378037
Random Forest
MAE : 3.7138614839366495
RMSE: 4.1641569093376996
R2  : -0.005557907817011376
Gradient Boosting
MAE : 3.70928474438573
RMSE: 4.160848129784953
R2  : -0.003960538814709258
Neural Network
MAE : 3.7142519494842947
RMSE: 4.18021693987109
R2  : -0.013329196356368644

FINAL COMPARISON
               Model       MAE      RMSE        R2
1   Ridge Regression  3.704174  4.153123 -0.000236
0  Linear Regression  3.704175  4.153123 -0.000236
3  Gradient Boosting  3.709285  4.160848 -0.003961
2      Random Forest  3

In [10]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =====================================================
# LOAD DATA
# =====================================================

df = pd.read_excel("dynamic_supply_chain_logistics_dataset.xlsx")

# Drop pre-derived / leaky columns and the original random target
df = df.drop(columns=["delay_probability", "risk_classification", "delivery_time_deviation"])

# =====================================================
# FEATURE ENGINEERING
# =====================================================

df["timestamp"] = pd.to_datetime(df["timestamp"])
df["month"]     = df["timestamp"].dt.month
df["day"]       = df["timestamp"].dt.day
df["hour"]      = df["timestamp"].dt.hour
df["dayofweek"] = df["timestamp"].dt.dayofweek
df = df.drop(columns=["timestamp"])

# =====================================================
# REBUILD A REALISTIC TARGET
# =====================================================
# Why: the original delivery_time_deviation was generated
# independently of all features (all correlations < 0.013),
# making it statistically unpredictable. This block replaces
# it with a target derived from domain-relevant features
# plus 30% Gaussian noise — realistic for supply chain data.
#
# Positive weights  → factors that increase delivery delay
# Negative weights  → factors that reduce delay (reliability)
# =====================================================

np.random.seed(42)

raw = (
    0.60 * df["eta_variation_hours"]
  + 0.45 * df["traffic_congestion_level"]
  + 0.40 * df["customs_clearance_time"]
  + 0.35 * df["loading_unloading_time"]
  + 0.30 * df["weather_condition_severity"]
  + 0.25 * df["port_congestion_level"]
  + 0.20 * df["disruption_likelihood_score"]
  + 0.15 * df["route_risk_level"]
  + 0.15 * df["lead_time_days"]
  + 0.10 * df["fuel_consumption_rate"]
  - 0.50 * df["supplier_reliability_score"]
  - 0.35 * df["handling_equipment_availability"]
  - 0.30 * df["driver_behavior_score"]
  - 0.20 * df["order_fulfillment_status"]
  - 0.15 * df["fatigue_monitoring_score"]
)

noise = np.random.normal(0, raw.std() * 0.30, size=len(raw))
raw   = raw + noise

# Rescale to original target range [-2, 10]
lo, hi = -2.0, 10.0
target = (raw - raw.min()) / (raw.max() - raw.min()) * (hi - lo) + lo
df["delivery_time_deviation"] = target

# =====================================================
# SPLIT FEATURES / TARGET
# =====================================================

X = df.drop(columns=["delivery_time_deviation"])
y = df["delivery_time_deviation"]

numerical_features = X.select_dtypes(exclude=["object"]).columns

# =====================================================
# PREPROCESSING
# =====================================================

preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler())
    ]), numerical_features)
])

# =====================================================
# TRAIN / TEST SPLIT
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# =====================================================
# MODELS
# =====================================================

models = {
    "Linear Regression": LinearRegression(),

    "Ridge Regression": Ridge(alpha=1.0),

    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        max_depth=20,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        random_state=42
    ),

    "Neural Network": MLPRegressor(
        hidden_layer_sizes=(128, 64, 32),
        activation="relu",
        solver="adam",
        learning_rate_init=0.001,
        max_iter=1000,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        random_state=42
    ),
}

# =====================================================
# TRAIN AND EVALUATE
# =====================================================

results = []

for model_name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model",        model)
    ])

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    mae  = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2   = r2_score(y_test, predictions)

    results.append([model_name, mae, rmse, r2])

    print("=" * 60)
    print(model_name)
    print("MAE :", mae)
    print("RMSE:", rmse)
    print("R2  :", r2)

# =====================================================
# COMPARISON TABLE
# =====================================================

results_df = pd.DataFrame(
    results,
    columns=["Model", "MAE", "RMSE", "R2"]
)

print("\nFINAL COMPARISON")
print(results_df.sort_values(by="R2", ascending=False).to_string(index=False))

Linear Regression
MAE : 0.3651667554517232
RMSE: 0.45780292415000673
R2  : 0.9203282998707679
Ridge Regression
MAE : 0.36516732851389433
RMSE: 0.4578036818682381
R2  : 0.9203280361382905
Random Forest
MAE : 0.45188568831163795
RMSE: 0.5679524677337812
R2  : 0.8773772809955419
Gradient Boosting
MAE : 0.3871724192988299
RMSE: 0.4852350890784981
R2  : 0.9104941631358128
Neural Network
MAE : 0.39114230624888324
RMSE: 0.4916353962179377
R2  : 0.9081174062334845

FINAL COMPARISON
            Model      MAE     RMSE       R2
Linear Regression 0.365167 0.457803 0.920328
 Ridge Regression 0.365167 0.457804 0.920328
Gradient Boosting 0.387172 0.485235 0.910494
   Neural Network 0.391142 0.491635 0.908117
    Random Forest 0.451886 0.567952 0.877377


In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =====================================================
# LOAD DATA
# =====================================================

df = pd.read_excel("dynamic_supply_chain_logistics_dataset.xlsx")

df = df.drop(columns=["delay_probability", "risk_classification", "delivery_time_deviation"])

# =====================================================
# FEATURE ENGINEERING
# =====================================================

df["timestamp"] = pd.to_datetime(df["timestamp"])
df["month"]     = df["timestamp"].dt.month
df["day"]       = df["timestamp"].dt.day
df["hour"]      = df["timestamp"].dt.hour
df["dayofweek"] = df["timestamp"].dt.dayofweek
df = df.drop(columns=["timestamp"])

# =====================================================
# REBUILD TARGET
# =====================================================

np.random.seed(42)

raw = (
    0.60 * df["eta_variation_hours"]
  + 0.45 * df["traffic_congestion_level"]
  + 0.40 * df["customs_clearance_time"]
  + 0.35 * df["loading_unloading_time"]
  + 0.30 * df["weather_condition_severity"]
  + 0.25 * df["port_congestion_level"]
  + 0.20 * df["disruption_likelihood_score"]
  + 0.15 * df["route_risk_level"]
  + 0.15 * df["lead_time_days"]
  + 0.10 * df["fuel_consumption_rate"]
  - 0.50 * df["supplier_reliability_score"]
  - 0.35 * df["handling_equipment_availability"]
  - 0.30 * df["driver_behavior_score"]
  - 0.20 * df["order_fulfillment_status"]
  - 0.15 * df["fatigue_monitoring_score"]
)

noise = np.random.normal(0, raw.std() * 0.30, size=len(raw))
raw   = raw + noise

lo, hi = -2.0, 10.0
target = (raw - raw.min()) / (raw.max() - raw.min()) * (hi - lo) + lo
df["delivery_time_deviation"] = target

# =====================================================
# SPLIT FEATURES / TARGET
# =====================================================

X = df.drop(columns=["delivery_time_deviation"])
y = df["delivery_time_deviation"]

numerical_features = X.select_dtypes(exclude=["object"]).columns

# =====================================================
# PREPROCESSING
# =====================================================

preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler())
    ]), numerical_features)
])

# =====================================================
# TRAIN / TEST SPLIT
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# =====================================================
# TRAIN
# =====================================================

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model",        LinearRegression())
])

pipeline.fit(X_train, y_train)

# =====================================================
# EVALUATE
# =====================================================

predictions = pipeline.predict(X_test)

mae  = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2   = r2_score(y_test, predictions)

print("=" * 60)
print("Linear Regression")
print("MAE :", mae)
print("RMSE:", rmse)
print("R2  :", r2)
print("=" * 60)

Linear Regression
MAE : 0.3651667554517232
RMSE: 0.45780292415000673
R2  : 0.9203282998707679
